In [61]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression,Ridge
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor, VotingRegressor,StackingRegressor


import warnings
warnings.filterwarnings('ignore')


In [62]:
df = pd.read_csv("bangladesh_student_performance.csv")
display(df.head())

,date,gender,age,address,famsize,Pstatus,M_Edu,F_Edu,M_Job,F_Job,relationship,smoker,tuition_fee,time_friends,ssc_result,hsc_result
0,29/04/2018,M,18,Rural,GT3,Together,3,2,At_home,Farmer,No,No,71672,4,4.22,3.72
1,29/04/2018,F,19,Rural,LE3,Apart,0,4,Other,Health,Yes,No,26085,5,3.47,2.62
2,29/04/2018,F,19,Rural,GT3,Together,0,3,Teacher,Services,No,No,40891,3,3.32,2.56
3,29/04/2018,F,19,Rural,LE3,Apart,2,3,At_home,Business,No,No,50600,2,4.57,4.17
4,29/04/2018,M,17,Rural,GT3,Together,1,1,At_home,Farmer,No,No,62458,2,4.50,3.94


In [63]:

from ydata_profiling import ProfileReport
profile = ProfileReport(df,title='Bangladesh Student Performance Prediction', explorative=True)
profile.to_file("bangladesh_student_performance_report.html")

Summarize dataset:   0%|          | 0/5 [00:00<?, ?it/s]

100%|██████████| 16/16 [00:00<00:00, 339.75it/s]


Generate report structure:   0%|          | 0/1 [00:00<?, ?it/s]

Render HTML:   0%|          | 0/1 [00:00<?, ?it/s]

Export report to file:   0%|          | 0/1 [00:00<?, ?it/s]

In [64]:
df.columns

Index(['date', 'gender', 'age', 'address', 'famsize', 'Pstatus', 'M_Edu',
       'F_Edu', 'M_Job', 'F_Job', 'relationship', 'smoker', 'tuition_fee',
       'time_friends', 'ssc_result', 'hsc_result'],
      dtype='object')

In [65]:
df.columns = df.columns.str.strip().str.lower()
df.columns

Index(['date', 'gender', 'age', 'address', 'famsize', 'pstatus', 'm_edu',
       'f_edu', 'm_job', 'f_job', 'relationship', 'smoker', 'tuition_fee',
       'time_friends', 'ssc_result', 'hsc_result'],
      dtype='object')

In [66]:
df.drop(columns=['date'],inplace=True)

In [67]:

#^ correlation for numerical values
corr_target = df.select_dtypes(include=np.number).corr()['hsc_result'].sort_values(ascending=False)
corr_target


hsc_result      1.000000
ssc_result      0.950178
m_edu           0.063776
f_edu           0.054811
tuition_fee     0.038068
age            -0.009857
time_friends   -0.156356
Name: hsc_result, dtype: float64

In [68]:

#^ Separate X and y
X = df.drop(columns=['hsc_result'],axis=1) # means axis=1 for columns
y = df['hsc_result']



In [69]:

num_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),  
    ('scaler', StandardScaler())
])

In [70]:
cat_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore'))
])

In [71]:
# Define numerical and categorical features
numerical_features = X.select_dtypes(include=['int64', 'float64']).columns.tolist()
categorical_features = X.select_dtypes(include=['object']).columns.tolist()

# combine them
preprocessor = ColumnTransformer(
    transformers=[
        ("num", num_transformer, numerical_features),
        ("cat", cat_transformer, categorical_features),
    ]
)

In [72]:

X_train,X_test,y_train,y_test = train_test_split(X,y,test_size=0.2,random_state=42)

X_train.shape, X_test.shape, y_train.shape, y_test.shape

((1614, 14), (404, 14), (1614,), (404,))

In [73]:

#& Base Learner
reg_lr = LinearRegression()
reg_rf = RandomForestRegressor(n_estimators=100, random_state=42)
reg_gb = GradientBoostingRegressor(n_estimators=100, random_state=42)


In [74]:

#& Voting regressor
voting_reg = VotingRegressor(estimators=[
    ('lr', reg_lr),
    ('rf', reg_rf),
    ('gb', reg_gb)
])

In [75]:
# ^ Stacking
stacking_reg = StackingRegressor(
    estimators=[
        ("lr", reg_lr),
        ("rf", reg_rf)
    ],
    final_estimator=Ridge() #* the meta learner
)

### Model Training  

In [76]:

#^ dict of all model
model_to_train       =          {
    'Linear Regression': reg_lr,
    'Random Forest': reg_rf,
    'Gradient Boasting': reg_gb,
    'Voting Ensemble': voting_reg,
    'Stacking Ensemble': stacking_reg
}

In [77]:

#& Training and Evaluation
results = []

for name, model in model_to_train.items():
    # create full pipeline with preprocessor and model
    pipe = Pipeline(
        [
            ('preprocessor', preprocessor),
            ('model', model)
        ]
    )

    # Train
    pipe.fit(X_train, y_train)

    # Predict
    y_pred = pipe.predict(X_test)

    # Evaluate
    mse = mean_squared_error(y_test, y_pred)
    r2 = r2_score(y_test, y_pred)
    rmse = np.sqrt(mse)
    mae = np.mean(np.abs(y_test - y_pred))

    results.append({
        'model': name,
        'mse': mse,
        'r2': r2,
        'rmse': rmse,
        'mae': mae
    })

results_df = pd.DataFrame(results).sort_values(by='r2', ascending=False)
results_df

,model,mse,r2,rmse,mae
2,Gradient Boasting,0.015155,0.959565,0.123107,0.098902
3,Voting Ensemble,0.015919,0.957528,0.126169,0.100838
4,Stacking Ensemble,0.016980,0.954697,0.130307,0.103853
1,Random Forest,0.018647,0.950248,0.136556,0.108201
0,Linear Regression,0.020269,0.945920,0.142371,0.111376


In [78]:

#^ Visualize the results
best_model = results_df.iloc[0]['model']
best_model_obj = model_to_train[best_model]
print(f"Best model based on R2 score: {best_model}")

final_pipe = Pipeline(
    [
        ('preprocessor', preprocessor),
        ('model', best_model_obj)
    ]
)

final_pipe.fit(X_train, y_train)
y_final_pred = final_pipe.predict(X_test)




Best model based on R2 score: Gradient Boasting


In [79]:
plt.figure(figsize=(8, 6))
sns.scatterplot(x=y_test, y=y_final_pred)
plt.plot([y.min(), y.max()], [y.min(), y.max()], 'r--')  # Line for perfect predictions
plt.xlabel('Actual HSC Result')
plt.ylabel('Predicted HSC Result')
plt.title(f'Actual vs Predicted HSC Result ({best_model})')
plt.show()



In [80]:

#^ Cross Validation
from sklearn.model_selection import cross_val_score
rf_pipe = Pipeline(
    [
        ('preprocessor', preprocessor),
        ('model', reg_rf)
    ]
)

# 5 fold cross validation
cv_scores = cross_val_score(rf_pipe, X_train, y_train, cv=5, scoring='neg_mean_squared_error')
cv_rmse = np.sqrt(-cv_scores)
print(f"Cross-validated RMSE for Random Forest: {cv_rmse.mean():.4f} ± {cv_rmse.std():.4f}")


Cross-validated RMSE for Random Forest: 0.1422 ± 0.0083


In [81]:

#& Stacking pipeline with cross validation
stacking_pipe = Pipeline(
    [
        ('preprocessor', preprocessor),
        ('model', stacking_reg)
    ]
)

cv_scores_stacking = cross_val_score(stacking_pipe, X_train, y_train, cv=5, scoring='neg_mean_squared_error',n_jobs=-1)
cv_rmse_stacking = np.sqrt(-cv_scores_stacking)
print(f"Cross-validated RMSE for Stacking Regressor: {cv_rmse_stacking.mean():.4f} ± {cv_rmse_stacking.std():.4f}")


Cross-validated RMSE for Stacking Regressor: 0.1319 ± 0.0066


In [83]:

# Grid Search for Hyperparameter Tuning
rf_pipe = Pipeline(
    [
        ('preprocessor', preprocessor),
        ('model', reg_rf)
    ]
)

param_grid = {
    'model__n_estimators': [100, 200],
    'model__max_depth': [None, 10, 20],
    'model__min_samples_split': [2, 5],
    'model__min_samples_leaf': [1, 2]
}

In [84]:

from sklearn.model_selection import GridSearchCV
grid_search = GridSearchCV(estimator=rf_pipe, param_grid=param_grid, cv=3, scoring='neg_mean_squared_error', n_jobs=-1)
grid_search.fit(X_train, y_train)

print(f"Best parameters for Random Forest: {grid_search.best_params_}")
best_rf = grid_search.best_estimator_
y_rf_pred = best_rf.predict(X_test)


Best parameters for Random Forest: {'model__max_depth': None, 'model__min_samples_leaf': 2, 'model__min_samples_split': 5, 'model__n_estimators': 200}


In [85]:

# Randomized search 
from sklearn.model_selection import RandomizedSearchCV
from scipy.stats import randint
param_dist = {
    'model__n_estimators': randint(100, 500),
    'model__max_depth': [None] + list(range(10, 50, 10)),
    'model__min_samples_split': randint(2, 10),
    'model__min_samples_leaf': randint(1, 5)
}

random_search = RandomizedSearchCV(estimator=rf_pipe, param_distributions=param_dist, n_iter=20, cv=3, scoring='neg_mean_squared_error', n_jobs=-1, random_state=42)
random_search.fit(X_train, y_train)

print(f"Best parameters from Randomized Search: {random_search.best_params_}")
best_rf_random = random_search.best_estimator_
y_rf_random_pred = best_rf_random.predict(X_test)


Best parameters from Randomized Search: {'model__max_depth': 20, 'model__min_samples_leaf': 3, 'model__min_samples_split': 4, 'model__n_estimators': 187}


In [88]:

#! Save Model
import pickle

from sklearn.linear_model import LinearRegression

X_train = [[1, 2], [3, 4], [5, 6]]
y_train = [1, 2, 3]
model = LinearRegression()
model.fit(X_train, y_train)

filename = 'linear_model.pkl'
with open(filename, 'wb') as file:
    pickle.dump(model, file)
    

#^ Load Model
with open(filename, 'rb') as file:
    loaded_model = pickle.load(file)
        
loaded_model.predict([[7, 8]])


array([4.])

In [89]:

#! Save Random Forest Model

with open('best_rf_model.pkl', 'wb') as file:
    pickle.dump(best_rf, file)


# ML Flow

In [1]:
!pip install mlflow

Defaulting to user installation because normal site-packages is not writeable
  Using cached mlflow-3.10.0-py3-none-any.whl.metadata (31 kB)
  Using cached mlflow_skinny-3.10.0-py3-none-any.whl.metadata (32 kB)
  Using cached mlflow_tracing-3.10.0-py3-none-any.whl.metadata (19 kB)
  Using cached flask_cors-6.0.2-py3-none-any.whl.metadata (5.3 kB)
  Using cached docker-7.1.0-py3-none-any.whl.metadata (3.8 kB)
  Using cached graphene-3.4.3-py2.py3-none-any.whl.metadata (6.9 kB)
  Using cached huey-2.6.0-py3-none-any.whl.metadata (4.3 kB)
  Using cached skops-0.13.0-py3-none-any.whl.metadata (5.6 kB)
  Using cached waitress-3.0.2-py3-none-any.whl.metadata (5.8 kB)
  Using cached fastapi-0.133.0-py3-none-any.whl.metadata (30 kB)
  Using cached opentelemetry_api-1.39.1-py3-none-any.whl.metadata (1.5 kB)
  Using cached opentelemetry_proto-1.39.1-py3-none-any.whl.metadata (2.3 kB)
  Using cached opentelemetry_sdk-1.39.1-py3-none-any.whl.metadata (1.5 kB)
  Using cached uvicorn-0.41.0-py3-none

  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
streamlit 1.3

In [3]:
import mlflow

